# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
dupes_count = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
log('1. Drop Duplicates', 'Removed duplicate rows', dupes_count)

[1. Drop Duplicates] Removed duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [4]:
df['price'] = df['price'].astype(str).str.replace('$', '', regex=False).astype(float)
log('2. Clean Price', 'Converted price to float', len(df))
print((df['qty'] * df['price']).sum())

[2. Clean Price] Converted price to float (300 row(s))
4594.5


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
initial_len = len(df)
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')
df = df[df['qty'] > 0].copy()
df['qty'] = df['qty'].astype(int)

rows_dropped = initial_len - len(df)
log('3. Clean Qty', 'Dropped rows with missing or non-positive qty', rows_dropped)

print((df['qty'] * df['price']).sum())

[3. Clean Qty] Dropped rows with missing or non-positive qty (25 row(s))
4740.0


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho'
}

df['item'] = df['item'].map(ITEM_MAP)
log('4. Canonicalize Item', 'Standardized item names into 3 distinct products', len(df))

[4. Canonicalize Item] Standardized item names into 3 distinct products (275 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
CAT_MAP = {
    'Food': 'Food',
    'food': 'Food',
    'Merch': 'Merch',
    'Apparel': 'Merch',  # Business Decision: Consolidation into Merch
    'RainGear': 'Rain Gear',
    'rain-gear': 'Rain Gear'
}

df['category'] = df['category'].map(CAT_MAP)
log('5. Normalize Category', 'Standardized categories and merged Apparel into Merch', len(df))

[5. Normalize Category] Standardized categories and merged Apparel into Merch (275 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert df.duplicated().sum() == 0, "Duplicates found!"
assert df['qty'].min() >= 1, "Invalid quantity found!"
assert df['price'].dtype == float, "Price is not float!"

assert df['item'].nunique() == 3, "Unexpected number of items!"
assert set(df['category'].unique()) == {'Food', 'Merch', 'Rain Gear'}, "Unexpected category values!"

print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
df['revenue'] = df['qty'] * df['price']

cat_rev = df.groupby('category')['revenue'].sum().sort_values(ascending=False).round(2)
total_rev = round(df['revenue'].sum(), 2)

print("--- Revenue by Category ---")
print(cat_rev)
print("\nTotal Revenue:", f"${total_rev:,.2f}")

--- Revenue by Category ---
category
Food         1656.0
Merch        1572.0
Rain Gear    1512.0
Name: revenue, dtype: float64

Total Revenue: $4,740.00


**What I would tell the vendor:** _..._ Stock more Food items, as Food generated the highest overall revenue ($1,656.00) among all product categories.

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,1. Drop Duplicates,Removed duplicate rows,15
1,2. Clean Price,Converted price to float,300
2,3. Clean Qty,Dropped rows with missing or non-positive qty,25
3,4. Canonicalize Item,Standardized item names into 3 distinct products,275
4,5. Normalize Category,Standardized categories and merged Apparel int...,275


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.


**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.


a.) Step 3 changed the revenue total the most. Before the step revenue was $4594.50 and it rose to $4,740 after.

b.)Step 5 consolidated Apparel into Merch which was a decision that resonable could have made differently. You could keep the two values separate as their own categories. Leaving them separate would show that Merch isnt the seond biggest revenue category and rain-gear would be greater.